# Estrazione Codice Civile da Normattiva

Notebook per estrarre tutti gli articoli del Codice Civile da Normattiva in CSV e validare il risultato.

Output principale: `src/data/statutes/codice_civile_normattiva.csv`.

Schema output:
- `article_id`
- `article_title`
- `article_text`
- `article_references` (lista JSON di riferimenti interni al codice)
- `external_references` (lista JSON di riferimenti esterni)
- `source`
- `libro_codice_civile`

Integrazione Neo4j:
- `src/db/db_orchestrator.py` gestisce sia `article_references`/`external_references` sia `reference`/`external_reference`.
- I riferimenti interni vengono normalizzati e usati per creare relazioni `(:Statute)-[:CITES]->(:Statute)` intra-codice.

- I riferimenti sono normalizzati con correzioni di suffisso e whitelist interna: se un articolo non esiste nel codice, viene spostato in `external_reference`/`external_references`.


In [3]:
from __future__ import annotations

import csv
import json
import html
import http.cookiejar
import re
import urllib.request
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

BASE_NORMATTIVA_URL = "https://www.normattiva.it/uri-res/N2Ls?urn:nir:stato:regio.decreto:1942-03-16;262"
OUTPUT_CSV_PATH = Path("../src/data/statutes/codice_civile_normattiva.csv")
RAW_DEBUG_DIR = Path("./tmp_normattiva_cc")
SAVE_RAW_HTML = False
INCLUDE_UPDATES = False
INCLUDE_PRELIMINARY_PROVISIONS = False

# Robustezza runtime (evita perdere tutto in caso di blocchi rete/interruzione manuale)
REQUEST_TIMEOUT_SECONDS = 30
REQUEST_RETRIES = 4
REQUEST_BASE_SLEEP_SECONDS = 0.8
REQUEST_THROTTLE_SECONDS = 0.03
SAVE_CHECKPOINT_EVERY = 150
RESUME_FROM_EXISTING_OUTPUT = False
PROGRESS_EVERY = 100

ARTICLE_LINK_RE = re.compile(
    r"onclick=\"return showArticle\('(/atto/caricaArticolo\?[^']+)'\s*,\s*this\);\"[^>]*class=\"numero_articolo\">\s*art\.\s*([^<]+?)\s*</a>",
    re.IGNORECASE,
)
ART_REF_RE = re.compile(r"\bart(?:t|icolo)?\.?\s*(\d+(?:-[a-z]+)*(?:\.\d+)?)", re.IGNORECASE)
EXTERNAL_REFERENCE_MARKERS = (
    "c.p.",
    "codice penale",
    "c.p.p",
    "codice di procedura penale",
    "c.p.c",
    "codice di procedura civile",
    "cost.",
    "costituzione",
    "decreto",
    "d.lgs",
    "d.l.",
    "dpr",
    "legge",
)


@dataclass
class CivilArticleEntry:
    article_id: str
    article_title: str
    article_text: str
    article_references: str
    external_references: str
    source: str
    libro_codice_civile: str


def _build_opener() -> urllib.request.OpenerDirector:
    jar = http.cookiejar.CookieJar()
    opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(jar))
    opener.addheaders = [("User-Agent", "Mozilla/5.0")]
    return opener


def _normalize_article_label(raw_label: str) -> str:
    label = raw_label.strip().lower()
    label = label.replace("‑", "-").replace("–", "-").replace("—", "-")
    label = re.sub(r"\s+", "-", label)
    label = label.strip(".")
    return label


_SUFFIX_ORDER = {
    "bis": 1,
    "ter": 2,
    "quater": 3,
    "quinquies": 4,
    "sexies": 5,
    "septies": 6,
    "octies": 7,
    "novies": 8,
    "decies": 9,
    "undecies": 10,
    "duodecies": 11,
    "terdecies": 12,
    "quaterdecies": 13,
    "quinquiesdecies": 14,
    "sexiesdecies": 15,
    "septiesdecies": 16,
    "duodevicies": 17,
    "vicies": 18,
}


def _token_sort_key(token: str) -> tuple[int, int | str]:
    if token.isdigit():
        return (2, int(token))
    order = _SUFFIX_ORDER.get(token)
    if order is not None:
        return (1, order)
    return (3, token)


def _article_sort_key(label: str) -> tuple[int, tuple[tuple[int, int | str], ...], str]:
    normalized = _normalize_article_label(re.sub(r"^art", "", label, flags=re.IGNORECASE))
    m = re.match(r"^(\d+)(.*)$", normalized)
    if not m:
        return (10**9, tuple(), normalized)
    base = int(m.group(1))
    rest = m.group(2).strip("-./")
    if not rest:
        return (base, tuple(), "")
    tokens = [t for t in re.split(r"[-./]", rest) if t]
    token_keys = tuple(_token_sort_key(t) for t in tokens)
    return (base, token_keys, rest)



def _html_to_lines(fragment: str) -> list[str]:
    text = re.sub(r"(?is)<script.*?>.*?</script>", " ", fragment)
    text = re.sub(r"(?is)<style.*?>.*?</style>", " ", text)
    text = re.sub(r"(?i)<br\s*/?>", "\n", text)
    text = re.sub(r"(?i)</div>|</p>|</li>|</h\d>", "\n", text)
    text = re.sub(r"(?is)<[^>]+>", " ", text)
    text = html.unescape(text)
    lines = [ln.strip() for ln in text.splitlines()]
    return [ln for ln in lines if ln]


def _clean_title(raw_title: str) -> str:
    title = raw_title.strip()
    title = re.sub(r"^[\(\)\.\s]+", "", title)
    title = re.sub(r"[\(\)\.\s]+$", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    return title


def _infer_libro_codice_civile(article_label: str) -> str:
    m = re.match(r"^(\d+)", article_label)
    if not m:
        return "Fuori range"
    n = int(m.group(1))
    if 1 <= n <= 455:
        return "Libro I"
    if 456 <= n <= 809:
        return "Libro II"
    if 810 <= n <= 1172:
        return "Libro III"
    if 1173 <= n <= 2059:
        return "Libro IV"
    if 2060 <= n <= 2642:
        return "Libro V"
    if 2643 <= n <= 2969:
        return "Libro VI"
    return "Fuori range"


REF_SUFFIX_CORRECTIONS = {
    "nonies": "novies",
    "sexiesdecies": "sexdecies",
}


def _normalize_reference_suffixes(token: str) -> str:
    parts = token.split("-")
    if len(parts) <= 1:
        return token
    fixed = [parts[0]]
    for part in parts[1:]:
        fixed.append(REF_SUFFIX_CORRECTIONS.get(part, part))
    return "-".join(fixed)


def _normalize_reference_article(raw_ref: str) -> str:
    token = raw_ref.strip().lower()
    token = token.replace("‑", "-").replace("–", "-").replace("—", "-")
    m = re.search(r"(\d+(?:-[a-z]+)*(?:\.\d+)?)", token)
    if not m:
        return ""
    return _normalize_reference_suffixes(m.group(1))


def _extract_references(text: str, valid_internal_refs: set[str] | None = None) -> tuple[list[str], list[str]]:
    if not text:
        return [], []

    internal: set[str] = set()
    external: set[str] = set()

    for match in ART_REF_RE.finditer(text):
        ref = _normalize_reference_article(match.group(1))
        if not ref:
            continue

        w_start = max(0, match.start() - 64)
        w_end = min(len(text), match.end() + 96)
        window = text[w_start:w_end].lower()

        is_internal_cc = ("c.c." in window) or ("codice civile" in window)
        is_external = any(marker in window for marker in EXTERNAL_REFERENCE_MARKERS)

        if is_external and not is_internal_cc:
            label = "Art. " + ref
            if "c.p." in window or "codice penale" in window:
                label += " c.p."
            elif "c.p.p" in window or "codice di procedura penale" in window:
                label += " c.p.p."
            elif "c.p.c" in window or "codice di procedura civile" in window:
                label += " c.p.c."
            elif "cost" in window:
                label += " Cost."
            external.add(label)
            continue

        if valid_internal_refs is not None and ref not in valid_internal_refs:
            external.add("Art. " + ref)
            continue

        internal.add(ref)

    return sorted(internal, key=_article_sort_key), sorted(external)


def _extract_title_and_body(lines: list[str], article_label: str) -> tuple[str, str]:
    if not lines:
        return "", ""

    heading_pattern = re.escape(article_label)
    heading_pattern = heading_pattern.replace("-", r"[-\s]?")
    heading_pattern = heading_pattern.replace(r"\.", r"\.?")
    art_re = re.compile(rf"^Art\.\s*{heading_pattern}\.?", re.IGNORECASE)

    art_idx = 0
    for i, line in enumerate(lines):
        if art_re.search(line):
            art_idx = i
            break

    content = lines[art_idx + 1 :] if art_idx + 1 < len(lines) else []
    if not content:
        return "", ""

    raw_title = content[0]
    title = _clean_title(raw_title)

    raw_title_stripped = raw_title.strip()
    looks_like_body = (
        not raw_title_stripped.startswith("(")
        and bool(re.match(r"^(Chiunque|Fuori|Quando|Qualora|La|Le|Il|I|Non|Nei|Nel|Se|Ciascuno|Ogni)\b", title, re.IGNORECASE))
        and len(title) > 60
    )

    if looks_like_body:
        title = ""
        body_lines = content
    else:
        body_lines = content[1:]

    if not INCLUDE_UPDATES:
        cut_idx = None
        for i, ln in enumerate(body_lines):
            if ln.upper().startswith("AGGIORNAMENTO") or ln.startswith("------------"):
                cut_idx = i
                break
        if cut_idx is not None:
            body_lines = body_lines[:cut_idx]

    body = " ".join(body_lines).strip()
    body = re.sub(r"\s+", " ", body).strip()
    body = re.sub(r"^\.\s*", "", body)
    return title, body



def _open_with_retry(
    opener: urllib.request.OpenerDirector,
    req: urllib.request.Request,
    retries: int = REQUEST_RETRIES,
    base_sleep: float = REQUEST_BASE_SLEEP_SECONDS,
    timeout: int = REQUEST_TIMEOUT_SECONDS,
) -> str:
    last_exc = None
    for attempt in range(retries):
        try:
            return opener.open(req, timeout=timeout).read().decode("utf-8", "ignore")
        except Exception as exc:
            last_exc = exc
            if attempt == retries - 1:
                raise
            time.sleep(base_sleep * (attempt + 1))
    raise last_exc


def _load_existing_rows(path: Path) -> dict[str, CivilArticleEntry]:
    if not path.exists():
        return {}

    existing: dict[str, CivilArticleEntry] = {}
    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            article_id = str(row.get("article_id", "")).strip()
            if not article_id:
                continue
            existing[article_id] = CivilArticleEntry(
                article_id=article_id,
                article_title=str(row.get("article_title", "")),
                article_text=str(row.get("article_text", "")),
                article_references=str(row.get("article_references", "[]")),
                external_references=str(row.get("external_references", "[]")),
                source=str(row.get("source", "codice_civile")),
                libro_codice_civile=str(row.get("libro_codice_civile", "")),
            )
    return existing


def extract_codice_civile(
    output_csv_path: Path = OUTPUT_CSV_PATH,
    resume_from_existing: bool = RESUME_FROM_EXISTING_OUTPUT,
) -> tuple[list[CivilArticleEntry], int]:
    opener = _build_opener()
    root_html = opener.open(
        BASE_NORMATTIVA_URL, timeout=REQUEST_TIMEOUT_SECONDS
    ).read().decode("utf-8", "ignore")

    if SAVE_RAW_HTML:
        RAW_DEBUG_DIR.mkdir(parents=True, exist_ok=True)
        (RAW_DEBUG_DIR / "root.html").write_text(root_html, encoding="utf-8")

    article_paths: dict[str, str] = {}
    for m in ARTICLE_LINK_RE.finditer(root_html):
        ajax_path = html.unescape(m.group(1))
        if not INCLUDE_PRELIMINARY_PROVISIONS and "art.flagTipoArticolo=1" in ajax_path:
            # Skip "Disposizioni sulla legge in generale" (preleggi)
            continue
        label = _normalize_article_label(html.unescape(m.group(2)))
        if label not in article_paths:
            article_paths[label] = ajax_path

    sorted_items = sorted(article_paths.items(), key=lambda x: _article_sort_key(x[0]))
    valid_internal_refs = set(article_paths.keys())

    existing_by_id: dict[str, CivilArticleEntry] = {}
    if resume_from_existing and output_csv_path.exists():
        existing_by_id = _load_existing_rows(output_csv_path)
        if existing_by_id:
            print(
                f"♻️ Resume attivo: {len(existing_by_id)} articoli già presenti in {output_csv_path}"
            )

    rows: list[CivilArticleEntry] = list(existing_by_id.values())
    done_labels = {row.article_id.removeprefix("art") for row in rows}

    added_since_checkpoint = 0
    extracted_now = 0

    for idx, (articolo_label, ajax_path) in enumerate(sorted_items, start=1):
        if articolo_label in done_labels:
            continue

        req = urllib.request.Request(
            "https://www.normattiva.it" + ajax_path,
            headers={
                "User-Agent": "Mozilla/5.0",
                "X-Requested-With": "XMLHttpRequest",
                "Referer": BASE_NORMATTIVA_URL,
                "Accept": "*/*",
            },
        )

        try:
            payload = _open_with_retry(opener, req)
        except KeyboardInterrupt:
            print("\n⏹️ Interruzione manuale ricevuta.")
            if output_csv_path and rows:
                save_csv(rows, output_csv_path)
                print(f"💾 Checkpoint salvato in: {output_csv_path.resolve()}")
            return rows, len(article_paths)
        except Exception as exc:
            print(
                f"⚠️ [{idx}/{len(sorted_items)}] Errore fetch Art. {articolo_label}: {exc}"
            )
            continue

        if REQUEST_THROTTLE_SECONDS > 0:
            time.sleep(REQUEST_THROTTLE_SECONDS)

        if SAVE_RAW_HTML and idx <= 20:
            (RAW_DEBUG_DIR / f"article_{idx:04d}_{articolo_label}.html").write_text(
                payload, encoding="utf-8"
            )

        start = payload.find('<div class="bodyTesto">')
        end = payload.find('<div class="d-flex justify-content-between', start)
        if start == -1 or end == -1:
            continue

        body_fragment = payload[start:end]
        lines = _html_to_lines(body_fragment)
        titolo, testo = _extract_title_and_body(lines, articolo_label)

        if not testo and lines:
            if len(lines) > 1:
                testo = " ".join(lines[1:]).strip()
                testo = re.sub(r"\s+", " ", testo)
                testo = re.sub(r"^\.\s*", "", testo)

        internal_refs, external_refs = _extract_references(testo, valid_internal_refs=valid_internal_refs)

        rows.append(
            CivilArticleEntry(
                article_id=f"art{articolo_label}",
                article_title=titolo,
                article_text=testo,
                article_references=json.dumps(internal_refs, ensure_ascii=False),
                external_references=json.dumps(external_refs, ensure_ascii=False),
                source="codice_civile",
                libro_codice_civile=_infer_libro_codice_civile(articolo_label),
            )
        )
        done_labels.add(articolo_label)
        extracted_now += 1
        added_since_checkpoint += 1

        if extracted_now % PROGRESS_EVERY == 0:
            print(
                f"   progresso: +{extracted_now} nuovi articoli (totale attuale: {len(rows)})"
            )

        if output_csv_path and SAVE_CHECKPOINT_EVERY > 0 and added_since_checkpoint >= SAVE_CHECKPOINT_EVERY:
            save_csv(rows, output_csv_path)
            print(f"   💾 checkpoint: {len(rows)} articoli salvati")
            added_since_checkpoint = 0

    return rows, len(article_paths)


def save_csv(rows: list[CivilArticleEntry], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "article_id",
                "article_title",
                "article_text",
                "article_references",
                "external_references",
                "source",
                "libro_codice_civile",
            ],
        )
        w.writeheader()
        sorted_rows = sorted(rows, key=lambda r: _article_sort_key(r.article_id))
        for row in sorted_rows:
            w.writerow(row.__dict__)

print(REQUEST_TIMEOUT_SECONDS, REQUEST_RETRIES, SAVE_CHECKPOINT_EVERY, RESUME_FROM_EXISTING_OUTPUT)
import inspect
print(inspect.signature(extract_codice_civile))

rows, normattiva_unique_articles = extract_codice_civile(
    output_csv_path=OUTPUT_CSV_PATH,
    resume_from_existing=RESUME_FROM_EXISTING_OUTPUT,
)
save_csv(rows, OUTPUT_CSV_PATH)

print(f"Timestamp UTC: {datetime.now(timezone.utc).isoformat()}")

for sample in ["art1", "art2", "art2043", "art2056", "art1223", "art1227", "art2697"]:
    item = next((r for r in rows if r.article_id == sample), None)
    if item:
        print(f"- {sample}: titolo='{item.article_title[:80]}' | incipit='{item.article_text[:120]}'")


30 4 150 False
(output_csv_path: 'Path' = PosixPath('../src/data/statutes/codice_civile_normattiva.csv'), resume_from_existing: 'bool' = False) -> 'tuple[list[CivilArticleEntry], int]'
   progresso: +100 nuovi articoli (totale attuale: 100)
   💾 checkpoint: 150 articoli salvati
   progresso: +200 nuovi articoli (totale attuale: 200)
   progresso: +300 nuovi articoli (totale attuale: 300)
   💾 checkpoint: 300 articoli salvati
   progresso: +400 nuovi articoli (totale attuale: 400)
   💾 checkpoint: 450 articoli salvati
   progresso: +500 nuovi articoli (totale attuale: 500)
   progresso: +600 nuovi articoli (totale attuale: 600)
   💾 checkpoint: 600 articoli salvati
   progresso: +700 nuovi articoli (totale attuale: 700)
   💾 checkpoint: 750 articoli salvati
   progresso: +800 nuovi articoli (totale attuale: 800)
   progresso: +900 nuovi articoli (totale attuale: 900)
   💾 checkpoint: 900 articoli salvati
   progresso: +1000 nuovi articoli (totale attuale: 1000)
   💾 checkpoint: 1050 art

In [4]:
import csv
import re
from collections import Counter

OUT = OUTPUT_CSV_PATH
with OUT.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print("Righe:", len(rows))
print("Colonne:", rows[0].keys())

ids = [(r.get("article_id") or "").strip() for r in rows]
dup = [(k, v) for k, v in Counter(ids).items() if v > 1]
print("Duplicati article_id:", len(dup))

bad_id = [x for x in ids if not re.match(r"^art\d+(?:(?:-[a-z]+(?:\.\d+)?)|(?:/\d+)|(?:\.\d+))?$", x)]
print("article_id formato anomalo:", len(bad_id))
if bad_id:
    print("Esempi:", bad_id[:20])

missing_title = sum(1 for r in rows if not (r.get("article_title") or "").strip())
missing_text = sum(1 for r in rows if not (r.get("article_text") or "").strip())
print("Titolo vuoto:", missing_title)
print("Testo vuoto:", missing_text)

if missing_title:
    print("Articoli con titolo vuoto (sample):")
    samples = [r.get("article_id") for r in rows if not (r.get("article_title") or "").strip()][:10]
    print(samples)

leading_dot = sum(1 for r in rows if (r.get("article_text") or "").lstrip().startswith("."))
paren_title = sum(
    1
    for r in rows
    if (r.get("article_title") or "").lstrip().startswith("((")
    or (r.get("article_title") or "").lstrip().startswith("( (")
)
print("Testo con leading dot:", leading_dot)
print("Titolo con artefatto parentesi:", paren_title)

print("Copertura albero Normattiva:", f"{len(rows)}/{normattiva_unique_articles}")

for a in ["art1", "art2", "art3", "art2043", "art2056", "art1223", "art1225", "art1226", "art1227", "art2697"]:
    row = next((r for r in rows if (r.get("article_id") or "").strip() == a), None)
    print(f"\n{a}:", "FOUND" if row else "MISSING")
    if row:
        print("  titolo:", (row.get("article_title") or "")[:100])
        print("  incipit:", (row.get("article_text") or "")[:130])

libri = Counter((r.get("libro_codice_civile") or "").strip() for r in rows)
print("\nDistribuzione libri:")
for k, v in sorted(libri.items()):
    print(f"- {k}: {v}")


Righe: 3230
Colonne: dict_keys(['article_id', 'article_title', 'article_text', 'article_references', 'external_references', 'source', 'libro_codice_civile'])
Duplicati article_id: 0
article_id formato anomalo: 0
Titolo vuoto: 1
Testo vuoto: 0
Articoli con titolo vuoto (sample):
['art342']
Testo con leading dot: 0
Titolo con artefatto parentesi: 0
Copertura albero Normattiva: 3230/3230

art1: FOUND
  titolo: Capacità giuridica
  incipit: La capacità giuridica si acquista dal momento della nascita. I diritti che la legge riconosce a favore del concepito sono subordin

art2: FOUND
  titolo: Maggiore età. Capacità di agire
  incipit: ((La maggiore età è fissata al compimento del diciottesimo anno. Con la maggiore età si acquista la capacità di compiere tutti gli

art3: FOUND
  titolo: ARTICOLO ABROGATO DALLA L. 8 MARZO 1975, N. 39
  incipit: ((ARTICOLO ABROGATO DALLA L. 8 MARZO 1975, N. 39 ))

art2043: FOUND
  titolo: Risarcimento per fatto illecito
  incipit: Qualunque fatto doloso o colp